## **Loan Prediction Machine Learning Model**

In [1]:
# Import Data Manipulation Libraries
import pandas as pd
import numpy as np
# Import Data Visualization Libraries
import seaborn as sns
import matplotlib.pyplot as plt
# Import Logging
import logging
logging.basicConfig(level = logging.INFO,
                    filename = 'logs/model.log',
                    filemode = 'w',
                    format = '%(name)s - %(levelname)s - %(message)s -%(levelname)s',
                    force = True)

# Import FilterWarning Libraries
import warnings
warnings.filterwarnings(action = 'ignore')
# Import Machine Learning Libraries
from sklearn.preprocessing import MinMaxScaler,RobustScaler,LabelEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy.stats.mstats import winsorize

In [2]:
!pip install "flaml[automl]"


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install mlflow


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Step1: Data Ingestion
def data_ingestion():
  df = pd.read_csv(r'C:\Devesh ITV\Machine Learning\Loan_Prediction_Classification_Model\data\loan_data.csv')
  df.shape
  return df

In [5]:
# Checking Dataset Information
# df.info()

In [6]:
'''
Note:
1. For Binary Classification: Sigmoid as activation function
2. For MultiClass Classification: Softmax as activation function
3. For Regression: Linear as activation function
4. For Binary Classification: BinaryCrossEntropy as loss function
5. For MultiClass Classification: CategoricalCrossEntropy as loss function
6. For Regression: MeanSquaredError as loss function
7. For Binary Classification: Accuracy as metric
8. For MultiClass Classification: Accuracy as metric
9. For Regression: MeanSquaredError as metric
10. Relu as activation function is used to avoid vanishing gradient problem and which impoved model performance in deep learning.

'''

'\nNote:\n1. For Binary Classification: Sigmoid as activation function\n2. For MultiClass Classification: Softmax as activation function\n3. For Regression: Linear as activation function\n4. For Binary Classification: BinaryCrossEntropy as loss function\n5. For MultiClass Classification: CategoricalCrossEntropy as loss function\n6. For Regression: MeanSquaredError as loss function\n7. For Binary Classification: Accuracy as metric\n8. For MultiClass Classification: Accuracy as metric\n9. For Regression: MeanSquaredError as metric\n10. Relu as activation function is used to avoid vanishing gradient problem and which impoved model performance in deep learning.\n\n'

In [7]:
# Step2: Data Preprocessing
from imblearn.over_sampling import SMOTE

def preprocessing(df):

  # Remove Duplicates from Dataset
  df.drop_duplicates()
  # Missing Values Imputation
  # df.isnull().sum().plot(kind = 'barh')
  '''
  - In the given dataset, No Missing values are found.
  -----------------------------------------------------
  To Prevent Data Leackage Following Steps are Recommended
  - Split the Dataset into X and y
  - Segregate Categorical Columns and Numerical Columns
  - Split the data into train and test : i.e. Seen Data and Unseen Data
  - Use SMOTE Technique to balance the dataset (Only For Classification Model)
  - Use Scaling Technique
  - Ready for Model Building

  '''

  # Segregate Categorical and Numerical Columns
  categorical_data = df.select_dtypes(include = 'object')
  numerical_data = df.select_dtypes(exclude = 'object')

  # Using Winsorization Technique : To Cap the Outliers
  for i in numerical_data.columns:
    df[i] = winsorize(df[i],limits = [0.05,0.05])

  # Encoding Categorical Data

  le = LabelEncoder()
  for i in categorical_data.columns:
    df[i] = le.fit_transform(df[i])

  # Split the Datatset into X and y
  X = df.drop(columns = ['loan_status'],axis = 1)
  y = df['loan_status']

  # Split the Dataset into Train and Test i.e. Seen Data and Unseen Data
  X_train,X_test,y_train,y_test = train_test_split(X,y,
                                                  test_size = 0.3,
                                                  random_state = 1)

  # Use SMOTE Technique

  sm = SMOTE(random_state = 1)
  X_train,y_train = sm.fit_resample(X_train,y_train)

  # Use Scaling Technique

  sc = MinMaxScaler()
  X_train = sc.fit_transform(X_train) # Seen Data
  X_test = sc.transform(X_test)       # Unseen Data

  print(y_train.value_counts())

  return X_train,X_test,y_train,y_test

In [8]:
from flaml import AutoML
import mlflow
import mlflow.sklearn
import numpy as np

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


def model_build(X_train, X_test, y_train, y_test):

    automl = AutoML()

    mlflow.set_experiment("FLAML_Classification")

    with mlflow.start_run(run_name="FLAML_Classification_Run") as run:

        automl.fit(
            X_train,
            y_train,
            task="classification",
            time_budget=30
        )

        y_pred = automl.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)

        precision = precision_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )

        print("Best Model:", automl.best_estimator)
        print("Best Parameters:", automl.best_config)
        print("Best Loss:", automl.best_loss)

        print("\nAccuracy:", accuracy)
        print("Precision:", precision)
        print("Recall:", recall)
        print("F1 Score:", f1)

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, y_pred))

        mlflow.log_param(
            "best_estimator",
            automl.best_estimator
        )

        mlflow.log_param(
            "time_budget",
            30
        )

        mlflow.log_params(
            automl.best_config
        )

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        mlflow.sklearn.log_model(
            automl.model,
            name="model",
            serialization_format="cloudpickle"
        )

        run_id = run.info.run_id

        print("\nMLflow Run ID:", run_id)

    model_uri = f"runs:/{run_id}/model"

    loaded_model = mlflow.sklearn.load_model(model_uri)

    loaded_prediction = loaded_model.predict(X_test)

    print("\nMLflow Prediction:")
    print(loaded_prediction)

    print(
        "\nPrediction Match:",
        np.array_equal(y_pred, loaded_prediction)
    )

    return automl, loaded_model

In [9]:
def main():
  df = data_ingestion()
  X_train,X_test,y_train,y_test = preprocessing(df)
  model_build(X_train,X_test,y_train,y_test)

main()

loan_status
0    24579
1    24579
Name: count, dtype: int64


2026/08/19 16:54:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/19 16:54:00 INFO mlflow.store.db.utils: Updating database tables
2026/08/19 16:54:04 INFO mlflow.tracking.fluent: Experiment with name 'FLAML_Classification' does not exist. Creating a new experiment.


[flaml.automl.logger: 08-19 16:54:04] {2375} INFO - task = classification
[flaml.automl.logger: 08-19 16:54:04] {2386} INFO - Evaluation method: holdout
[flaml.automl.logger: 08-19 16:54:04] {2489} INFO - Minimizing error metric: 1-roc_auc
[flaml.automl.logger: 08-19 16:54:04] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 08-19 16:54:04] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 08-19 16:54:05] {3046} INFO - Estimated sufficient time budget=8679s. Estimated necessary time budget=201s.
[flaml.automl.logger: 08-19 16:54:05] {3097} INFO -  at 0.3s,	estimator lgbm's best error=5.9799e-02,	best estimator lgbm's best error=5.9799e-02
[flaml.automl.logger: 08-19 16:54:05] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 08-19 16:54:05] {3097} INFO -  at 0.7s,	estimator lgbm's best error=5.0669e-02,	best estimator lgbm's best error=5.0669e-02
[flaml

MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['collections.OrderedDict', 'flaml.automl.contrib.histgb.HistGradientBoostingEstimator', 'flaml.automl.data.DataTransformer', 'flaml.automl.model.CatBoostEstimator', 'flaml.automl.model.ElasticNetEstimator', 'flaml.automl.model.ExtraTreesEstimator', 'flaml.automl.model.KNeighborsEstimator', 'flaml.automl.model.LGBMEstimator', 'flaml.automl.model.LRL1Classifier', 'flaml.automl.model.LRL2Classifier', 'flaml.automl.model.LassoLarsEstimator', 'flaml.automl.model.RandomForestEstimator', 'flaml.automl.model.SGDEstimator', 'flaml.automl.model.SVCEstimator', 'flaml.automl.model.SparkAFTSurvivalRegressionEstimator', 'flaml.automl.model.SparkGBTEstimator', 'flaml.automl.model.SparkGLREstimator', 'flaml.automl.model.SparkLGBMEsti